<a href="https://colab.research.google.com/github/jiewenTsai/U1FAF6-project/blob/main/docs/20260319-Untitled7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 長格式 (lme4) 和寬格式 (lavaan) 的估計是相同的

- 2026-03-19
- jw

In [ ]:
system('sudo apt install r-cran-lavaan r-cran-lme4')

In [ ]:
library(lavaan)
library(lme4)
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ tidyr::expand() masks Matrix::expand()
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
✖ tidyr::pack()   masks Matrix::pack()
✖ tidyr::unpack() masks Matrix::unpack()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [ ]:
data_wide <- HolzingerSwineford1939[c(1, 7:9)]

In [ ]:
head(data_wide)

,id,x1,x2,x3
,<int>,<dbl>,<dbl>,<dbl>
1,1,3.333333,7.75,0.375
2,2,5.333333,5.25,2.125
3,3,4.500000,5.25,1.875
4,4,5.333333,7.75,3.000
5,5,4.833333,4.75,0.875
6,6,5.333333,5.00,2.250


In [ ]:
data_long <- pivot_longer(data_wide, cols = -id )

In [ ]:
  head(data_long)

id,name,value
<int>,<chr>,<dbl>
1,x1,3.333333
1,x2,7.750000
1,x3,0.375000
2,x1,5.333333
2,x2,5.250000
2,x3,2.125000


In [ ]:
## wide data (sem)

model <- "
F =~ 1*x1 + 1*x2 + 1*x3
x1 ~~ resvar*x1
x2 ~~ resvar*x2
x3 ~~ resvar*x3

x1 ~ 0*1
x2 ~ 1
x3 ~ 1
F ~ 1
"

fit_sem <- sem(model, data_wide, meanstructure=T)
summary(fit_sem)

lavaan 0.6-10 ended normally after 23 iterations

  Estimator                                         ML
  Optimization method                           NLMINB
  Number of model parameters                         7
  Number of equality constraints                     2
                                                      
  Number of observations                           301
                                                      
Model Test User Model:
                                                      
  Test statistic                                 7.422
  Degrees of freedom                                 4
  P-value (Chi-square)                           0.115

Parameter Estimates:

  Standard errors                             Standard
  Information                                 Expected
  Information saturated (h1) model          Structured

Latent Variables:
                   Estimate  Std.Err  z-value  P(>|z|)
  F =~                                                
    x

In [ ]:
## data long

fit_lmer <- lmer(value ~ (1|id) + name, data_long, REML=F)
summary(fit_lmer)

Linear mixed model fit by maximum likelihood  ['lmerMod']
Formula: value ~ (1 | id) + name
   Data: data_long

      AIC       BIC    logLik -2*log(L)  df.resid 
   2731.4    2755.4   -1360.7    2721.4       898 

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-2.94955 -0.62068 -0.02622  0.55642  2.58217 

Random effects:
 Groups   Name        Variance Std.Dev.
 id       (Intercept) 0.4794   0.6924  
 Residual             0.8589   0.9268  
Number of obs: 903, groups:  id, 301

Fixed effects:
            Estimate Std. Error t value
(Intercept)  4.93577    0.06668   74.02
namex2       1.15227    0.07554   15.25
namex3      -2.68535    0.07554  -35.55

Correlation of Fixed Effects:
       (Intr) namex2
namex2 -0.566       
namex3 -0.566  0.500

### 固定效果：觀察 lavaan- Intercepts 部分 vs lme4- Fixed effects 部分

結構方程式估計如下

```
Intercepts:
                   Estimate  Std.Err  z-value  P(>|z|)
   .x1                0.000                           
   .x2                1.152    0.076   15.253    0.000
   .x3               -2.685    0.076  -35.547    0.000
    F                 4.936    0.067   74.021    0.000
```

多層次模式估計如下：

```
Fixed effects:
            Estimate Std. Error t value
(Intercept)  4.93577    0.06668   74.02
namex2       1.15227    0.07554   15.25
namex3      -2.68535    0.07554  -35.55
```


### 隨機效果：觀察 lavaan- Variances 部分 vs lme4- Random effects: 部分


結構方程式估計如下

```
Variances:
                   Estimate  Std.Err  z-value  P(>|z|)
   .x1      (rsvr)    0.859    0.050   17.349    0.000
   .x2      (rsvr)    0.859    0.050   17.349    0.000
   .x3      (rsvr)    0.859    0.050   17.349    0.000
    F                 0.479    0.065    7.426    0.000
```

多層次模式估計如下：

```
Random effects:
 Groups   Name        Variance Std.Dev.
 id       (Intercept) 0.4794   0.6924  
 Residual             0.8589   0.9268  
Number of obs: 903, groups:  id, 301
```



